# Run All Evaluations

This notebook runs `evaluation.run_all_evals` instead of launching `eval_router`, `eval_NLU`, `eval_DM`, and `eval_NLG` one by one.

The project folder is found automatically in Google Drive by searching for project marker files. This works both for folders in `MyDrive` and for shared folders added as shortcuts to Drive.

Before running the full evaluation, make sure the project contains the latest versions of the evaluation scripts.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")


def find_project_dir() -> Path:
    marker_files = [
        "colab_utils/requirements_colab.txt",
        "colab_utils/colab_launcher.py",
        "app/chatbot.py",
    ]

    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for requirements_file in root.rglob("requirements_colab.txt"):
            candidate = requirements_file.parents[1]

            if all((candidate / marker).exists() for marker in marker_files):
                candidates.append(candidate)

    candidates = sorted(set(candidates), key=lambda path: str(path).lower())

    if not candidates:
        raise FileNotFoundError(
            "Project folder not found. If the folder was shared with you, "
            "open Google Drive, go to 'Shared with me', right-click the project folder, "
            "choose 'Add shortcut to Drive', then run this cell again."
        )

    if len(candidates) == 1:
        return candidates[0]

    print("Multiple project folders found:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}. {candidate}")

    choice = input("Choose the project folder number, or press Enter to use the first one: ").strip()
    if not choice:
        return candidates[0]

    return candidates[int(choice) - 1]


PROJECT_DIR = find_project_dir()
COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)

In [ ]:
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH" 

In [ ]:
import os
import sys
import importlib
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not found. Public models can still be loaded, but loading may fail.")

os.environ["APP_DEBUG"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s:%(message)s",
    force=True,
)

print("Project configured.")

In [ ]:
from pathlib import Path

required_files = [
    "evaluation/run_all_evals.py",
    "evaluation/eval_router.py",
    "evaluation/eval_NLU.py",
    "evaluation/eval_DM.py",
    "evaluation/eval_NLG.py",
]

missing = [path for path in required_files if not Path(path).exists()]

if missing:
    print("Missing files:")
    for path in missing:
        print("-", path)
else:
    print("All evaluation files found.")

print("\nEvaluation folder:")
!ls -lah evaluation

## Select models and components

By default, the notebook tries to read all available models from `llm.config.MODELS`.

If you want a quick test, replace `MODELS_TO_EVALUATE` with a shorter list, for example:

```python
MODELS_TO_EVALUATE = ["qwen3_4b"]
```


In [ ]:
import importlib

try:
    config_module = importlib.import_module("llm.config")
    MODELS_TO_EVALUATE = list(config_module.MODELS.keys())
except Exception as exc:
    print("Could not automatically read llm.config.MODELS:", repr(exc))
    MODELS_TO_EVALUATE = ["qwen3_4b"]

COMPONENTS_TO_EVALUATE = ["router", "nlu", "dm", "nlg"]

# Lower values are safer on Colab when evaluating multiple models.
BATCH_SIZE = 1

print("Models to evaluate:", MODELS_TO_EVALUATE)
print("Components to evaluate:", COMPONENTS_TO_EVALUATE)
print("Batch size:", BATCH_SIZE)

## Run all evaluations

This cell launches:

```bash
python -m evaluation.run_all_evals --models ... --components router nlu dm nlg --batch-size ...
```

The script should load one model, evaluate all selected components, then move to the next model.


In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable,
    "-m",
    "evaluation.run_all_evals",
    "--models",
    *MODELS_TO_EVALUATE,
    "--components",
    *COMPONENTS_TO_EVALUATE,
    "--batch-size",
    str(BATCH_SIZE),
]

print("Running command:")
print(" ".join(cmd))

start = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

for line in process.stdout:
    print(line, end="")
    output_lines.append(line)

return_code = process.wait()
elapsed = time.time() - start

print(f"\nFinished in {elapsed / 60:.1f} minutes.")
print("Return code:", return_code)

if return_code != 0:
    print("\nLast 120 output lines:")
    print("=" * 80)
    print("".join(output_lines[-120:]))
    raise RuntimeError("run_all_evals failed. Read the output above.")

## Inspect generated result files

This cell searches for evaluation outputs such as:

- `router_results.json`
- `nlu_results.json`
- `dm_results.json`
- `nlg_results.json`
- `*_errors.json`
- `*_predictions.json`
- `nlg_manual_review.json`


In [ ]:
from pathlib import Path

json_files = sorted(Path("evaluation").rglob("*.json"))

interesting = [
    path for path in json_files
    if any(token in path.name for token in ["results", "errors", "predictions", "manual_review"])
]

print(f"Found {len(interesting)} evaluation output files.\n")

for path in interesting:
    print(path)

In [ ]:
import json
from pathlib import Path

result_files = sorted(Path("evaluation").rglob("*_results.json"))

for path in result_files:
    print("=" * 80)
    print(path)
    try:
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)
        print(json.dumps(data.get("metrics", data), indent=2, ensure_ascii=False)[:4000])
    except Exception as exc:
        print("Could not read file:", repr(exc))

## Create a ZIP with all evaluation outputs

Run this at the end if you want to download the generated evaluation files from Colab.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

zip_base = Path("evaluation_outputs")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir="evaluation")

print("Created:", zip_path)
files.download(zip_path)